In [ ]:
import sys

import os
import re
import click
import tqdm
import pickle
import numpy as np
import torch
import PIL.Image
import dnnlib
from torch_utils import distributed as dist
from PIL import Image
from tqdm import tqdm
from matplotlib import pyplot as plt
import torch.nn.functional as F

# project 
from project_utils.paths import PROJECT_ROOT, CONFIG_DIR, DATA_DIR
from pou import gen_weight_grid, f2p, p2f

# for video display
from IPython.display import HTML
import matplotlib.animation as animation
import matplotlib as mpl


In [ ]:


def animate_all_fields(fields, titles=None, n_start=0, n_frame_show=999, n_row=1,
                       vmin=None, vmax=None, fig_w=7, fig_h=4, fps=2, cmap='viridis', grid=True,
                       land_sea_mask=None, mask_color='Reds', mask_alpha=0.1,
                       save_path=None):
    """
    Animate all fields in a grid layout.
    
    Parameters:
    -----------
    fields : array-like
        Data array with shape (n_fields, n_frames, height, width)
    titles : str or list of str
        Either a single title string (used for all fields) or a list of titles 
        (one for each field, length should match n_fields)
    n_start : int
        Starting frame index (default: 0)
    n_frame_show : int
        Number of frames to show (default: 999)
    n_row : int
        Number of rows in the subplot grid. Columns are calculated automatically
        as ceil(n_fields / n_row). Any unfilled grid cells are hidden. (default: 1)
    vmin : float
        Minimum value for colormap (default: None)
    vmax : float
        Maximum value for colormap (default: None)
    fig_w : float
        Width per subplot column in inches (default: 7)
    fig_h : float
        Height per subplot row in inches (default: 4)
    fps : float
        Frames per second (default: 2)
    cmap : str
        Colormap name (default: 'viridis')
    grid : bool
        Whether to show grid lines on each subplot (default: True)
    land_sea_mask : array-like, optional
        2D mask array overlaid on each field (default: None)
    mask_color : str
        Colormap used to display the land/sea mask (default: 'Reds')
    mask_alpha : float
        Transparency of the land/sea mask overlay (default: 0.1)
    save_path : str, optional
        If provided, saves the animation to this path. Use a '.gif' extension 
        to save as GIF (requires Pillow), or any other extension for video 
        (requires ffmpeg). (default: None)
    
    Returns:
    --------
    HTML object containing the animation
    """
    nframes = np.min([n_frame_show, fields.shape[1] - n_start])
    total_duration = nframes / fps
    
    n_fields = fields.shape[0]
    
    
    # Generate default titles if not provided
    if titles is None:
        titles_list = [f"Image {i+1}" for i in range(n_fields)]
    elif isinstance(titles, str):
        # If titles is a single string, use it for name
        base_title = titles
        titles_list = [f"{base_title}" for i in range(n_fields)]
    elif len(titles) == 1:
        # If titles is a list with one element, use it as prefix
        base_title = titles[0]
        titles_list = [f"{base_title} ({i+1})" for i in range(n_fields)]
    elif len(titles) != n_fields:
        print(f"Warning: {len(titles)} titles provided for {n_fields} fields. Using defaults.")
        titles_list = [f"Image {i+1}" for i in range(n_fields)]
    else:
        titles_list = titles
    
    
    # Create figure with n_row rows, n_col columns
    n_col = int(np.ceil(n_fields / n_row))
    figsize = (n_col * fig_w, n_row * fig_h)
    fig, axes = plt.subplots(n_row, n_col, figsize=figsize)
    
    # Flatten axes to a 1D list for uniform indexing
    if n_row == 1 and n_col == 1:
        axes_flat = [axes]
    elif n_row == 1 or n_col == 1:
        axes_flat = list(axes.flatten())
    else:
        axes_flat = list(axes.flatten())
    
    # Hide any extra subplots (when n_fields doesn't fill the grid)
    for idx in range(n_fields, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # Plot initial frame for each field
    mats = []
    for field_idx in range(n_fields):
        mat = axes_flat[field_idx].imshow(fields[field_idx, n_start], vmax=vmax, vmin=vmin, cmap=cmap)
        if land_sea_mask is not None:
            axes_flat[field_idx].imshow(land_sea_mask, cmap=mask_color, alpha=mask_alpha)
        axes_flat[field_idx].set_title(f'{titles_list[field_idx]} (frame={n_start})')
        if grid:
            axes_flat[field_idx].grid(which='both', color='r', linestyle='--', linewidth=0.5, alpha=0.8)
        fig.colorbar(mat, ax=axes_flat[field_idx])
        mats.append(mat)
    
    fig.tight_layout()
    
    # Update function for animation
    def update(i):
        frame_idx = i + n_start
        for field_idx in range(n_fields):
            mats[field_idx].set_data(fields[field_idx, frame_idx])
            axes_flat[field_idx].set_title(f'{titles_list[field_idx]} (frame={frame_idx})')
        return mats
    
    # Create animation
    ani = animation.FuncAnimation(
        fig, update, frames=nframes, 
        interval=int(1000.0 * total_duration / nframes),
        blit=False
    )
    if save_path is not None:
        if save_path.endswith('.gif'):
            ani.save(save_path, writer='pillow', fps=fps)
        else:
            ani.save(save_path, writer='ffmpeg', fps=fps)
        print(f"Animation saved to {save_path}")
        
    plt.close(fig)
    
    # Return HTML animation
    return HTML(ani.to_jshtml())


In [ ]:
def edm_sampler(
    net, latents, conditions=None, class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
    # Adjust noise levels based on what's supported by the network.
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    # Time step discretization.
    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

    # Main sampling loop.
    x_next = latents.to(torch.float64) * t_steps[0]
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
        x_cur = x_next

        # Increase noise temporarily.
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

        # Euler step.
        denoised = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        # Apply 2nd order correction.
        if i < num_steps - 1:
            denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

    return x_next




In [ ]:
# load original data 128x128
# show SQG orignal fields
dataset_dir = os.path.join(DATA_DIR, "SQG") # root SQG data dir
fields_all = torch.load(os.path.join(dataset_dir, 'SQG_fields.pt'))
print('loaded raw fields: ', fields_all.shape)


In [ ]:

mpl.rcParams['animation.embed_limit'] = 30 # in MB (30 MB in this example)

n_frame_show = 50
n_start = 0 
nframes = n_frame_show
total_duration = n_frame_show * 0.1

def update(i):
	i = i + n_start
	mat0.set_data(fields_all[i,0])
	ax.set_title(f'SQG True')

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 5))
mat0 = ax.imshow(fields_all[n_start,0], vmax=1, vmin=-1)
fig.colorbar(mat0, ax=ax)
fig.tight_layout()



ani = animation.FuncAnimation(
	fig, update, frames=nframes, interval=int(1000.0 * total_duration / nframes)
)

# Save as mp4
ani.save('sqg_true.mp4', writer='ffmpeg', fps=10, dpi=150)

plt.close(fig)

# Display the animation inline
HTML(ani.to_jshtml())  # or use ani.to_html5_video()



In [ ]:
# device 
device=torch.device('cuda:0')


# Load network.
network_pkl = ''
network_pkl = '/home/neil/cond-diff-pou/outputs/00000-sqg_patch_c2-medium-gpus1-batch256-fp32/network-snapshot-002000.pkl'

print(f'Loading network from "{network_pkl}"...')
with dnnlib.util.open_url(network_pkl, verbose=True) as f:
    net_32_patch = pickle.load(f)['ema'].to(device)

print('condition channels: ',net_32_patch.img_cond_channels)

In [ ]:
def edm_sampler_pou(
    weight_grid_2d, net, latents, conditions=None, 
    class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
    # Adjust noise levels based on what's supported by the network.
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    # Time step discretization.
    step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
    t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

    # condition (BCHW)
    patch_cond = f2p(conditions) if conditions is not None else None
    weight_grid_flat = weight_grid_2d.view(-1, 1, weight_grid_2d.shape[2], weight_grid_2d.shape[3]).to(latents.device)

    # Main sampling loop.
    x_next = latents.to(torch.float64) * t_steps[0]
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
        x_cur = x_next

        # Increase noise temporarily.
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

        # Euler step.
        patch_x_hat = f2p(x_hat)
        patch_denoised = net(patch_x_hat, t_hat, patch_cond, class_labels).to(torch.float64) 
        denoised = p2f(patch_denoised, weight_grid_flat)


        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        # Apply 2nd order correction.
        if i < num_steps - 1:
            # denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
            
            patch_x_next = f2p(x_next)
            patch_denoised = net(patch_x_next, t_next, patch_cond, class_labels).to(torch.float64) 
            denoised = p2f(patch_denoised, weight_grid_flat)


            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

    return x_next

In [ ]:
# 128
# get starting frames 

n_pred_step = 50

patch_size = 32
expand_ratio = 4
batch_size = 5
output_size = patch_size * expand_ratio
weight_grid_2d = gen_weight_grid(patch_size, expand_ratio)

# condition generation
fields_dm = fields_all[0:2,0,:,:].to(device=device).expand(batch_size, -1, -1, -1)
print('initial conditioning:')
print(fields_dm.shape)


In [ ]:


# generation loop
for i in tqdm(range(n_pred_step), desc="Generating 128x128 turb fields"):
	latents = torch.randn([batch_size, 1, output_size, output_size], device=device)
	conditions = fields_dm[:,-2:,:,:]
	samples = edm_sampler_pou(weight_grid_2d, net_32_patch, latents, conditions, num_steps=18)
	fields_dm = torch.cat((fields_dm, samples), dim=1)

print(fields_dm.shape)

In [ ]:
sample_gen = fields_dm.cpu().numpy()
true_field = fields_all[:52,0,:,:].cpu().numpy()[None,...]
show_all = np.concatenate((true_field, sample_gen), axis=0)
print('show_all shape: ', show_all.shape)

In [ ]:
mpl.rcParams['animation.embed_limit'] = 100 # in MB (30 MB in this example)



# Assuming fields has shape (n_fields, n_frames h, w)
animate_all_fields(show_all, titles=['True', 'sample1', 'sample2', 'sample3', 'sample4', 'sample5'], 
                   n_start=0, n_frame_show=999, vmin=-1, vmax=1, fps=10, n_row=2,fig_w=5, fig_h=4,grid=False,
                   save_path='128_pou.mp4')

In [ ]:
# tyni
# get starting frames 

n_pred_step = 50

patch_size = 32
expand_ratio = 1
batch_size = 4
output_size = patch_size * expand_ratio
weight_grid_2d = gen_weight_grid(patch_size, expand_ratio)

# condition generation
fields_dm = fields_all[0:2,0,:32,:32].to(device=device).expand(batch_size, -1, -1, -1)
print('initial conditioning:')
print(fields_dm.shape)


In [ ]:


# generation loop
for i in tqdm(range(n_pred_step), desc="Generating 128x128 turb fields"):
	latents = torch.randn([batch_size, 1, output_size, output_size], device=device)
	conditions = fields_dm[:,-2:,:,:]
	samples = edm_sampler_pou(weight_grid_2d, net_32_patch, latents, conditions, num_steps=18)
	fields_dm = torch.cat((fields_dm, samples), dim=1)

print(fields_dm.shape)

In [ ]:

# Assuming fields has shape (n_fields, n_frames h, w)
animate_all_fields(fields_dm.cpu().numpy(), titles=['sample1', 'sample2', 'sample3', 'sample4'], 
                   n_start=0, n_frame_show=999, vmin=-1, vmax=1, fps=10, n_row=2,fig_w=5, fig_h=4,grid=False,
                   save_path='32_raw.mp4')